# **Project: E-Commerce Business Intelligence & Customer Analytics**

**Internship Program:** Python Data Analytics with AI

**Dataset:** [UCI Online Retail Dataset](https://archive.ics.uci.edu/dataset/352/online+retail)


# **Phase 1: Requirement Analysis & Business Problem**

- **Problem Statement:** The e-commerce store experiences high website traffic but faces challanges in converting visistors into high-value customers, improving customer retention, and maximising profit margins.

- **KPIs (Success Metrics):**
  - **Revenue Trends:** Analyze revenue over time (daily/monthly/seasonal) to identify peak sales period.
  - **Average Order Value (AOV):** Total Revenue / Number of Orders.
  - **Customer Segmentation (VIP Identification):** Identify high-value customers based on revenue contribution or RFM analysis.
  - **Customer Churn Risk:** Customer with no purchases in the last 90 days.
  - **Repeat Purchase Rate:** Percentage of customers who made more than one purchase.
  - **Product Performance:** Identify top-selling and high-revenue generating products.
  - **Customer Lifetime Value (CLV):** Estimate customer value based on historical total spending.

- **Goal:** To deliver 3 data-driven strategies based on customer behaviour and sales insights to increase overall revenue by 10% within the next 6 months.

# **Phase 2: Data Acquisition & Pre-processing**

### ***Approach:***

**1. Data Acquisition:**
- Load the Online Retail transactional dataset from UCI repository into the analysis environment.

**2. Data Cleaning:**
- Remove records with missing CustomerID (required for customer-level analysis)
- Filter out cancellation transactions (InvoiceNo starting with 'C') OR handle them separately as returns.
- Check for and remove duplicate records if present.
- Handle missing or irrelevant product descriptions.
- Filter out invalid values (e.g., zero or negative UnitPrice where applicable).
- Treat negative Quantity values as returns and exclude them from sales analysis.

**3. Data Transformation & Feature Engineering:**
- Covert data types (e.g., InvoiceDate to datetime format).
- Create TotalPrice = Quntity x UnitPrice.
- Extract time-based features (Month, Year, Day of week).
- Prepare dataset for customer-level aggrigation (for later RFM analysis).

**4. Data Storage:**
- Store the cleaned and transformed dataset into an SQL Database and into CSV file.
- This enables structured querying using SQL and simulates a real-world business intelligence environment.

## **Imports**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

## **Upload Dataset**

In [ ]:
from google.colab import files
uploaded = files.upload()

## **Load Dataset**

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"
df = pd.read_excel(url)
df.head()

## **Basic Inspection**

In [ ]:
df.info()
df.describe()
df.isnull().sum()

## **Data Cleaning**

In [ ]:
# Remove missing CustomerID
df = df.dropna(subset=['CustomerID'])

# Remove cancellation invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove duplicates
df = df.drop_duplicates()

# Remove missing Description
df = df.dropna(subset=['Description'])

# Remove invalid UnitPrice
df = df[df['UnitPrice'] > 0]

# Remove negative Quantity
df = df[df['Quantity'] > 0]

df.shape

## **Feature Engineering**

In [ ]:
# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# Create time features
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour

## **Final Preview**

In [ ]:
df.head()

## **Store in SQLite Database**

In [ ]:
conn = sqlite3.connect('ecommerce.db')

df.to_sql('retail', conn, if_exists='replace', index=False)

# Verify
query = "SELECT * FROM retail LIMIT 5"
pd.read_sql(query, conn)

## **Store CSV**

In [ ]:
df.to_csv('cleaned_retail.csv', index=False)

# **Phase 3: Exploratory Data Analysis (EDA)**

### ***Approach:***

We use SQL queries and visualization to explore patterns in sales, customer behaviour and product performance.

**1. Time-Series Analysis:**
- Analyze revenue trends over time (monthly, weekly, daily).
- Identify seasonality and peak sales period.
- Examine hourly transaction patterns to determine peak shopping times.
- **Business Insight:** Optimize marketing campaigns and promotion based on peak activity periods.

**2. Sales & Order Behaviour Analysis:**
- Analyze distribution of order quantities and unit prices.
- Evaluate Average Order Value (AOV) trends.
- Identify patterns in basket size (number of items per order) and purchasing behaviour.

**3. Product Performance Analysis:**
- Identify top-selling products by quantity and revenue.
- Detect underperforming products.
- **Business Insight:** Focus on high-performing products and optimize inventory.

**4. Customer Analysis:**
- Analyze customer purchase frequency.
- Identify high-value customers (VIP identification).
- Examine repeat vs one-time customers.
- **Business Insight:** Support customer segmentation and retention strategies.

**5. Geographical Analysis:**
- Analyze revenue contribution by country.
- Identify key markets driving sales.
- **Business Insight:** Support region-specific marketing and expansion strategies.

## **Monthly Revenue Trend**

In [ ]:
monthly_revenue = df.groupby(['Year','Month'])['TotalPrice'].sum().reset_index()

plt.figure()
sns.lineplot(data=monthly_revenue, x='Month', y='TotalPrice')
plt.title('Monthly Revenue Trend')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.show()

## **Hourly Shopping Pattern**

In [ ]:
hourly = df.groupby('Hour')['InvoiceNo'].count()

plt.figure()
hourly.plot(kind='bar')
plt.title('Hourly Shopping Pattern')
plt.xlabel('Hour')
plt.ylabel('Orders')
plt.show()

## **Quantity Distribution**

In [ ]:
plt.figure()
sns.histplot(df['Quantity'], bins=50)
plt.title('Quantity Distribution')
plt.show()

## **Unit Price Distribution**

In [ ]:
plt.figure()
sns.histplot(df['UnitPrice'], bins=50)
plt.title('Unit Price Distribution')
plt.show()

## **Average Order Value (AOV)**

In [ ]:
aov = df.groupby('InvoiceNo')['TotalPrice'].sum().mean()
print("Average Order Value:", aov)

## **Basket Size**

In [ ]:
basket = df.groupby('InvoiceNo')['Quantity'].sum()

plt.figure()
sns.histplot(basket, bins=50)
plt.title('Basket Size Distribution')
plt.show()

## **Top 10 Products**

In [ ]:
top_products = df.groupby('Description')['TotalPrice'].sum().sort_values(ascending=False).head(10)

plt.figure()
top_products.plot(kind='bar')
plt.title('Top 10 Products by Revenue')
plt.show()

## **Repeat vs One-time Customers**

In [ ]:
cust_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()

repeat = (cust_orders > 1).sum()
one_time = (cust_orders == 1).sum()

print("Repeat Customers:", repeat)
print("One-Time Customers:", one_time)

## **Revenue by Country**

In [ ]:
country = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).head(10)

plt.figure()
country.plot(kind='bar')
plt.title('Revenue by Country')
plt.show()

# **Phase 4: Customer Segmentation (RFM Analysis)**

### ***Approach:***

We apply the RFM (Recency, Frequency, Monetary) model to segment customers based on purchasing behavior and identify high-value and at-risk customers.

**1. RFM Calculation:**
- **Recency:** Number of days since the customer’s last purchase (lower is better).
- **Frequency:** Total number of transactions per customer (higher is better).
- **Monetary:** Total revenue generated per customer (higher is better).

**2. RFM Scoring & Segmentation:**
- Assign scores (e.g., 1–5) for each RFM metric using quantiles.
- Combine scores to create customer segments such as:
  - VIP / High-Value Customers
  - Loyal Customers
  - At-Risk Customers
  - Lost Customers

**3. Business Insights:**
- Identify top revenue-contributing customers (VIPs).
- Detect customers with high past value but low recent activity (churn risk).
- Support targeted marketing and retention strategies.

**4. Supporting Analysis:**

- **Correlation Analysis:**
  - Analyze relationships between key variables (e.g., Frequency vs Monetary, Quantity vs UnitPrice).

- **Outlier Detection:**
  - Use IQR method on order quantity and spending.
  - Identify potential wholesale customers whose behavior differs significantly from typical retail customers.
  - Adjust analysis if needed to avoid skewed insights.

## **Prepare Data**

In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency','Frequency','Monetary']
rfm.head()

## **RFM Scoring**

In [ ]:
rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1])
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5])

rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm.head()

## **Customer Segmentation**

In [ ]:
def segment(row):
    if row['RFM_Score'] == '555':
        return 'VIP'
    elif row['F'] >= 4:
        return 'Loyal'
    elif row['R'] <= 2:
        return 'At Risk'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment, axis=1)

rfm['Segment'].value_counts()

## **Top VIP Customers**

In [ ]:
vip = rfm[rfm['Segment'] == 'VIP'].sort_values(by='Monetary', ascending=False)
vip.head()

## **Correlation**

In [ ]:
rfm[['Recency','Frequency','Monetary']].corr()

## **Outlier Detection (IQR)**

In [ ]:
Q1 = df['Quantity'].quantile(0.25)
Q3 = df['Quantity'].quantile(0.75)

IQR = Q3 - Q1

outliers = df[(df['Quantity'] < (Q1 - 1.5*IQR)) | (df['Quantity'] > (Q3 + 1.5*IQR))]

outliers.shape

## **Save RFM**

In [ ]:
rfm.to_csv('rfm_data.csv')

# **Phase 5: Business Insights & Strategy Recommendations**   (Explaining Part - Briefly in Notebook)

In this phase, we translate analytical findings from EDA and RFM segmentation into actionable business strategies aligned with revenue and customer retention goals.

**1. Key Insights:**
- Top 10 Best-Selling Products for Revenue contribution (identified in EDA).
- Revenue shows strong seasonality with peak sales during high-demand months (identified in EDA).
- A small group of VIP customers contributes a significant portion of total revenue (RFM analysis).
- At-risk customers form a large recoverable segment with declining purchase activity.
- Certain countries (e.g., top-performing regions from EDA) dominate overall revenue contribution.

**2. Strategic Recommendations:**
  - **Customer Retention Strategy:**  
  Launch targeted campaigns for At-Risk customers using discounts or personalized email marketing.

- **Revenue Optimization Strategy:**  
  Focus on VIP customers with loyalty rewards and exclusive offers to increase retention.

- **AOV Growth Strategy:**  
  Implement cross-selling and bundling of frequently co-purchased products to increase basket size.

- **Marketing Optimization Strategy:**  
  Run campaigns during peak sales hours and high-performing months identified in EDA.

**3. Expected Impact:**
- Improved customer retention through reactivation of at-risk customers.
- Increased Average Order Value (AOV) via bundling and cross-selling.
- Higher revenue contribution from VIP customers through loyalty programs.
- Overall achievement of **~10% revenue growth target** within the defined period.

#### **Key Insights:**
- Revenue peaks in specific months indicating seasonality
- VIP customers contribute major revenue
- Some customers show churn risk
- Top products drive most sales

#### **Strategies:**
1. Target At-Risk customers with offers
2. Reward VIP customers with loyalty programs
3. Use product bundling to increase AOV

#### **Expected Impact:**
- Better retention
- Increased AOV
- Revenue growth up to 10%

# **Phase 6: Reporting & Dashboarding in Python**

In this phase, we present key business insights through a structured dashboard that summarizes revenue performance, customer behavior, and product analytics for stakeholders.

**1. Executive Summary Dashboard:**

The dashboard combines the most important business KPIs into a single visual view:

- Top 10 Best-Selling Products (Revenue Contribution)
- Monthly Revenue Trend (Seasonality Analysis)
- Customer Segmentation (RFM-based distribution)
- Peak Shopping Hours (Customer activity heatmap)

**2. Visual Storytelling:**
- Combine multiple visualizations into a single dashboard-style layout.
- Each plot represents a key business question:
  - What products generate the most revenue?
  - When does the business earn the most revenue?
  - Which customer segments drive profitability?
  - When are customers most active?

- Ensure all charts include:
  - Clear titles
  - Axis labels
  - Business-focused annotations (where necessary)

**3. Tools Used:**
- **Seaborn:** For statistical and distribution-based visualizations  
- **Matplotlib:** For layout control and multi-plot dashboard creation

**4. Optional Enhancement:**

For advanced interactivity and deeper exploration:

- Use **Plotly** for interactive visualizations  
- Use **Dash / Streamlit** to build a full interactive dashboard application

In [ ]:
plt.figure(figsize=(12,8))

# 1
plt.subplot(2,2,1)
top_products.plot(kind='bar')
plt.title('Top Products')

# 2
plt.subplot(2,2,2)
sns.lineplot(data=monthly_revenue, x='Month', y='TotalPrice')
plt.title('Monthly Revenue')

# 3
plt.subplot(2,2,3)
rfm['Segment'].value_counts().plot(kind='bar')
plt.title('Customer Segments')

# 4
plt.subplot(2,2,4)
hourly.plot(kind='bar')
plt.title('Peak Hours')

plt.tight_layout()
plt.show()

# **Phase 7: Project Handover & Deliverables**  (Doing Part - Not Notebook)

### ***Approach:***

Present the analysis, insights, and recommendations in a clear, concise, and business-focused format for stakeholders.

**1. Executive Summary:**
- Brief overview of the problem, approach, and key findings.
- Highlight 3–4 critical insights.

**2. Key Business Insights:**
- Revenue trends and seasonality.
- High-value (VIP) and at-risk customers.
- Top-performing products and markets.

**3. Strategic Recommendations:**
- Clearly outline 3 actionable strategies.
- Explain expected business impact (aligned with +10% revenue goal).

**4. Data Visualization:**
- Present dashboards and key charts created in Phase 6.
- Use storytelling to guide stakeholders through insights.

**5. Conclusion:**
- Summarize outcomes and potential next steps.
- Suggest future improvements (e.g., predictive modeling, personalization).

### **Conclusion:**
- Business can improve retention using segmentation
- Focus on high-value customers
- Improve marketing timing
- Future: add predictive models